# TeleComX — Customer Churn Analysis
## Notebook 4: Feature Engineering
**Analyst:** Adedayo Adeyinka A 
**Date:** June 2026  
**Objective:** Prepare data for predictive modelling by encoding, 
scaling, and splitting into training and testing sets.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Load clean data
df = pd.read_csv('../data/telecomx_clean.csv')

print("Data loaded:", df.shape)
print("Columns:", list(df.columns))

Data loaded: (7043, 21)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [2]:
# Drop customerID - it's just an identifier, not useful for modeling
df = df.drop('customerID', axis=1)

print("customerID removed.")
print("Remaining columns:", df.shape[1])

customerID removed.
Remaining columns: 20


In [3]:
# Convert Churn from Yes/No to 1/0
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

print("Churn value counts:")
print(df['Churn'].value_counts())

Churn value counts:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [4]:
# Columns with simple Yes/No values — convert to 1/0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

for col in binary_cols:
    df[col] = (df[col] == 'Yes').astype(int)

print("Binary columns encoded:")
print(df[binary_cols].head())

Binary columns encoded:
   Partner  Dependents  PhoneService  PaperlessBilling
0        1           0             0                 1
1        0           0             1                 0
2        0           0             1                 1
3        0           0             0                 0
4        0           0             1                 1


In [5]:
# Columns with more than 2 categories need get_dummies (one-hot encoding)
multi_cols = ['gender', 'MultipleLines', 'InternetService', 
              'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
              'TechSupport', 'StreamingTV', 'StreamingMovies',
              'Contract', 'PaymentMethod']

df = pd.get_dummies(df, columns=multi_cols, drop_first=True)

print("Encoding complete.")
print("New shape:", df.shape)

Encoding complete.
New shape: (7043, 31)


In [6]:
# Separate features (X) from target (y)
X = df.drop('Churn', axis=1)
y = df['Churn']

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print("Feature matrix shape:", X.shape)
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print(f"\nTraining churn rate: {y_train.mean()*100:.1f}%")
print(f"Testing churn rate:  {y_test.mean()*100:.1f}%")

Feature matrix shape: (7043, 30)
Training set: (5634, 30)
Testing set: (1409, 30)

Training churn rate: 26.5%
Testing churn rate:  26.5%


In [7]:
# Save feature matrix and target for use in modeling notebook
import numpy as np

np.save('../data/X_train.npy', X_train)
np.save('../data/X_test.npy', X_test)
np.save('../data/y_train.npy', y_train)
np.save('../data/y_test.npy', y_test)

# Save feature names for later interpretation
feature_names = list(X.columns)
pd.Series(feature_names).to_csv('../data/feature_names.csv', index=False)

print("Training and testing data saved.")
print(f"Features saved: {len(feature_names)}")

Training and testing data saved.
Features saved: 30
